# 🎬 keepsfx → giữ lại HIỆU ỨNG (SFX)

Bỏ giọng + nhạc khỏi video, giữ lại tiếng động/hiệu ứng → MP4 (audio = SFX) để lồng tiếng Việt.
Dùng **BandIt** (SOTA, license CC BY-NC — phi thương mại).

**Bước duy nhất:** chọn `Runtime → GPU T4 → Save`, rồi bấm ▶ chạy ô bên dưới (Accept Drive khi hỏi).

> Theo dõi các dòng `=== BUOC ... ===` để biết đang chạy tới đâu.

In [ ]:
print('=== BUOC 1: Mount Drive ===', flush=True)
from google.colab import drive
drive.mount('/content/drive')

import os, importlib.util, shutil
print('=== BUOC 2: Clone code ===', flush=True)
os.chdir('/content')
!rm -rf /content/keepsfx /content/bandit
!git clone -q https://github.com/yaylbanh/keepsfx.git
!git clone -q https://github.com/kwatcharasupat/bandit.git

print('=== BUOC 3: Cai thu vien (co the lau vai phut lan dau) ===', flush=True)
if not all(importlib.util.find_spec(m) for m in ['gradio', 'lightning', 'asteroid', 'fire', 'librosa']):
    !pip install -r /content/keepsfx/requirements.txt
    print('=== Cai xong thu vien ===', flush=True)
else:
    print('=== Thu vien da co, bo qua ===', flush=True)

print('=== BUOC 4: Chuan bi model BandIt ===', flush=True)
MODELS = '/content/drive/MyDrive/keepsfx_models'
CKPT_DIR = os.path.join(MODELS, 'ckpt')
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT = os.path.join(CKPT_DIR, 'dnr-3s-bark48-l1snr.ckpt')
OLD_CKPT = os.path.join(MODELS, 'dnr-3s-bark48-l1snr.ckpt')
if os.path.isfile(OLD_CKPT) and not os.path.isfile(CKPT):
    shutil.move(OLD_CKPT, CKPT)
    print('[*] Da di chuyen checkpoint cu vao ckpt/.', flush=True)
if not os.path.isfile(CKPT):
    print('[*] Tai checkpoint ~775MB ...', flush=True)
    !wget -q -O "{CKPT}" "https://zenodo.org/records/10160698/files/dnr-3s-bark48-l1snr.ckpt?download=1"
else:
    print('[*] Checkpoint da co, bo qua tai.', flush=True)
shutil.copyfile('/content/bandit/expt/dnr-3s-bark48-l1snr.yaml', os.path.join(MODELS, 'hparams.yaml'))

print('=== BUOC 5: Khoi dong app (UI se hien ben duoi) ===', flush=True)
os.environ['BANDIT_DIR'] = '/content/bandit'
os.environ['BANDIT_CKPT'] = CKPT
os.environ['KEEPSFX_OUTPUT'] = '/content/drive/MyDrive/keepsfx_output'
os.environ['KEEPSFX_SHARE'] = '0'   # 0 = UI trong o Colab (nhanh); 1 = co link gradio.live
os.chdir('/content/keepsfx')
%run app.py